In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [ ]:
main_url = "https://bookpansion.ru"

html = requests.get(
    main_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
).text

soup = BeautifulSoup(html, "html.parser")
links = soup.find_all("a", href=True)
urls = []

for a in links:
    title = a.get_text(" ", strip=True)
    href = a["href"]

    if "/pansionat/" in href and len(title) > 5:
        if href.startswith("/"):
            href = main_url + href
        urls.append({"title": title, "url": href})

df_urls = pd.DataFrame(urls)
df_urls = df_urls[
    ~df_urls["url"].str.contains("#reviews", na=False)
]
df_urls = df_urls.drop_duplicates(subset=["url"])

SERVICE_KEYWORDS = ["реабилитация", "деменция", "альцгеймер", "лежачие",
    "сиделка", "медицинский уход", "круглосуточный уход", "лфк", "массаж",
    "психолог", "диетолог", "5-разовое питание", "прогулки", "восстановление",
    "инсульт", "перелом", "уколы", "капельницы", "памперсы", "гигиена"]


results = []

for i, row in df_urls.iterrows():
    url = row["url"]
    print(f"{url}")

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )

        soup = BeautifulSoup(response.text, "html.parser")
        text = soup.get_text("\n", strip=True)
        lower_text = text.lower()

        title = row["title"]

        rating_match = re.search(r"(\d\.\d)\s*\d+\s*отзы", text)
        rating = (float(rating_match.group(1)) if rating_match else None)

        reviews_match = re.search(r"(\d+)\s*отзы", text)
        reviews = (int(reviews_match.group(1)) if reviews_match else None)

        prices = re.findall( r"от\s*\d[\d\s]*\s*(?:₽|руб)", text, re.I)
        prices = list(set([p.strip() for p in prices]))

        price_numeric = None
        if prices:
            nums = []
            for p in prices:
                num = re.sub(r"\D", "", p)
                if num:
                    nums.append(int(num))
            if nums:
                price_numeric = min(nums)


        found_services = []
        for service in SERVICE_KEYWORDS:
            if service.lower() in lower_text:
                found_services.append(service)

        dementia = "деменц" in lower_text
        alzheimer = "альцгеймер" in lower_text
        rehab = "реабил" in lower_text
        bedridden = "лежач" in lower_text

        results.append({
            "title": title,
            "url": url,
            "rating": rating,
            "reviews": reviews,

            "min_price_per_day": price_numeric,
            "all_prices": " | ".join(prices),

            "services": " | ".join(found_services),

            "dementia": dementia,
            "alzheimer": alzheimer,
            "rehab": rehab,
            "bedridden": bedridden
        })

        time.sleep(1)

    except Exception as e:

        print("ERROR:", e)


df = pd.DataFrame(results)
df = df.drop_duplicates(subset=["url"])
df.to_csv("competitors_pansionaty.csv", index=False, encoding="utf-8-sig")
